# Scenario 6 — Adversarial / Obfuscated Phishing Robustness
## Decoder Notebook (Qwen2.5-1.5B) — FIXED

**Hypothesis:** Decoder LLMs (semantic reasoning) are more robust to adversarial perturbations than Encoder models (token pattern matching).

**Fix applied:** Qwen is now LoRA fine-tuned on the same training data as BERT, making the comparison architecture-fair.
LoRA is required because full fine-tuning of 1.5B parameters exceeds the T4's 15GB VRAM.

**Why LoRA is still fair:**
- The hypothesis tests robustness (F1 drop), not absolute F1.
- The frozen base weights still provide Qwen's semantic reasoning — that's what we're testing.
- If Qwen still drops less despite partial adaptation, the hypothesis holds even more strongly.

**Limitation:** LoRA adapts fewer parameters than BERT's full fine-tuning. If results are inconclusive,
an A100 (40GB) with full fine-tuning would give a definitive answer.

**IMPORTANT:** Run the Encoder notebook first — this notebook loads X_te, X_adv, y_te from disk
to guarantee both notebooks use the exact same test and adversarial data.

In [ ]:
!nvidia-smi
!pip install -q "numpy==1.26.4" "scipy==1.12.0"
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate bitsandbytes peft datasets scikit-learn pandas tqdm

In [1]:
import os, re, time, warnings, random, string
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                           BitsAndBytesConfig, get_linear_schedule_with_warmup)
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW
from datasets import load_dataset
from huggingface_hub import HfFileSystem
from tqdm.auto import tqdm

# ── Reproducibility (must match encoder notebook) ────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

DECODER_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantisation to fit the base model weights in VRAM
BNB_CFG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# ── Shared evaluation function (identical to encoder notebook) ────────────────
def evaluate(y_true, y_pred, name=""):
    return {
        "Model":     name,
        "Accuracy":  f"{accuracy_score(y_true, y_pred):.4f}",
        "Precision": f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
        "Recall":    f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
        "F1":        f"{f1_score(y_true, y_pred, average='binary', zero_division=0):.4f}"
    }

Device: cuda
GPU: Tesla T4


In [2]:
# ── Load dataset and build train/test splits ─────────────────────────────────
print("Loading ealvaradob/phishing-dataset...")
fs = HfFileSystem()
all_files = fs.glob("datasets/ealvaradob/phishing-dataset/**")
texts_files = [f for f in all_files if "text" in f.lower() and f.endswith(".json")]
hf_url = "hf://" + texts_files[0]
ds = load_dataset("json", data_files={"train": hf_url}, split="train")
df = ds.to_pandas()
df["text"] = df["text"].astype(str).str.strip()
print(f"Total: {len(df):,} | Benign: {(df.label==0).sum():,} | Phishing: {(df.label==1).sum():,}")

X, y = df["text"].values, df["label"].values
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_te, y_val, y_te  = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print(f"Train {len(X_tr):,} | Val {len(X_val):,} | Test {len(X_te):,}")

Loading ealvaradob/phishing-dataset...


texts.json:   0%|          | 0.00/52.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Total: 20,137 | Benign: 12,465 | Phishing: 7,672
Train 14,095 | Val 3,021 | Test 3,021


In [3]:
# ── Generate adversarial test set ────────────────────────────────────────────
# random.seed is reset here so perturb() produces byte-for-byte identical
# outputs to the encoder notebook, regardless of how many random calls
# happened earlier in this notebook (dataset split, etc.)

random.seed(SEED)   # reset before perturbation

SYNONYMS = {
    "verify": "confirm",   "account": "profile",    "click": "select",
    "urgent": "important", "password": "credential", "login": "sign-in",
    "suspend": "restrict", "update": "refresh",      "confirm": "validate",
    "immediately": "promptly", "bank": "financial institution"
}
HOMOGLYPHS = {'a':'а','e':'е','o':'о','p':'р','c':'с','i':'і','x':'х'}

def perturb(text):
    words = text.split()
    for i, w in enumerate(words):
        if w.lower() in SYNONYMS and random.random() < 0.15:
            words[i] = SYNONYMS[w.lower()]
    text = " ".join(words)
    chars = list(text)
    for i in range(len(chars)):
        if chars[i].isalpha() and random.random() < 0.03:
            chars[i] = random.choice(string.ascii_lowercase)
    text = "".join(chars)
    words = text.split(); result = []
    for w in words:
        if len(w) > 4 and random.random() < 0.05:
            result.append(w[:len(w)//2] + " " + w[len(w)//2:])
        else:
            result.append(w)
    text = " ".join(result)
    return "".join(HOMOGLYPHS.get(c, c) if random.random() < 0.08 else c for c in text)

print("Generating adversarial test set...")
X_adv = np.array([perturb(t) for t in X_te])
print(f"Test samples: {len(X_te):,} | Adversarial samples: {len(X_adv):,}")
print("Original   :", X_te[0][:100])
print("Adversarial:", X_adv[0][:100])

Generating adversarial test set...
Test samples: 3,021 | Adversarial samples: 3,021
Original   : donnellan complete citation a summary of my request for the complete citation : keith s . donnellan 
Adversarial: dinnellan complеtе citation a summary of my request for the cоmplete citation : keith s . donnеllan 


In [4]:
# ── Dataset class for LoRA fine-tuning (CAUSAL_LM) ──────────────────────────
class QwenClfDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.enc = tokenizer(
            list(texts), padding="max_length", truncation=True,
            max_length=max_len, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return {k: v[i] for k, v in self.enc.items()}, self.labels[i]

In [6]:
# ── Load Qwen2.5-1.5B and attach LoRA adapters ───────────────────────────────
print("Loading Qwen2.5-1.5B-Instruct (4-bit)...")
tok = AutoTokenizer.from_pretrained(DECODER_MODEL_ID, trust_remote_code=True)
tok.pad_token = tok.eos_token
tok.padding_side = "left"   # causal LMs pad on the left

base_model = AutoModelForCausalLM.from_pretrained(
    DECODER_MODEL_ID,
    quantization_config=BNB_CFG,
    device_map="auto",
    trust_remote_code=True
)
base_model.config.pad_token_id = tok.eos_token_id

# TaskType.CAUSAL_LM — correct for a generative decoder like Qwen.
# SEQ_CLS causes a batch_size mismatch because PEFT routes labels through
# the causal LM loss which operates over every token, not per-sample.
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

# Token IDs for the two class labels — used in training and inference.
# Defined here because tok must exist first.
LABEL_IDS = {
    0: tok.encode(" legitimate", add_special_tokens=False)[0],
    1: tok.encode(" phishing",   add_special_tokens=False)[0]
}
print(f"Label token IDs — legitimate: {LABEL_IDS[0]} | phishing: {LABEL_IDS[1]}")

Loading Qwen2.5-1.5B-Instruct (4-bit)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
Label token IDs — legitimate: 22745 | phishing: 98097


In [7]:
# ── Fine-tune Qwen + LoRA on clean training data ─────────────────────────────
# We use standard causal LM loss but only supervise the final token position.
# The model learns to predict the correct label token at the end of each input.
print("Fine-tuning Qwen2.5-1.5B + LoRA on clean training data...")

tr_dl = DataLoader(
    QwenClfDataset(X_tr, y_tr, tok),
    batch_size=8, shuffle=True
)
opt   = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
              lr=3e-4, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, len(tr_dl) // 5, len(tr_dl) * 5)

for ep in range(5):
    model.train(); total_loss = 0
    for batch, lbl in tqdm(tr_dl, desc=f"Qwen LoRA ep{ep+1}/5"):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        lbl            = lbl.to(DEVICE)

        # Forward pass — get logits for every token position
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Take the logit at the LAST non-padding token for each sample,
        # then select only the two label token columns → (batch, 2)
        last_pos = attention_mask.sum(dim=1) - 1          # index of last real token
        last_logits = outputs.logits[                      # (batch, vocab)
            torch.arange(input_ids.size(0)), last_pos
        ]
        label_logits = last_logits[:, [LABEL_IDS[0], LABEL_IDS[1]]]  # (batch, 2)

        loss = torch.nn.functional.cross_entropy(label_logits, lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        total_loss += loss.item()
    print(f"  Epoch {ep+1} avg loss: {total_loss / len(tr_dl):.4f}")

print("Training complete.")

Fine-tuning Qwen2.5-1.5B + LoRA on clean training data...


Qwen LoRA ep1/5:   0%|          | 0/1762 [00:00<?, ?it/s]

  Epoch 1 avg loss: 1.2553


Qwen LoRA ep2/5:   0%|          | 0/1762 [00:00<?, ?it/s]

  Epoch 2 avg loss: 1.1573


...

  Epoch 3 avg loss: 1.1424


Qwen LoRA ep4/5:   0%|          | 0/1762 [00:00<?, ?it/s]

  Epoch 4 avg loss: 1.1346


Qwen LoRA ep5/5:   0%|          | 0/1762 [00:00<?, ?it/s]

  Epoch 5 avg loss: 1.1329
Training complete.


In [10]:
# ── Evaluate Qwen+LoRA on clean and adversarial test sets ────────────────────
def predict_decoder(model, tok, texts, batch=16):
    """Classify by comparing label token logits at the last real token position."""
    model.eval(); preds = []
    for i in range(0, len(texts), batch):
        enc = tok(
            list(texts[i:i+batch]), padding=True, truncation=True,
            max_length=256, return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            outputs  = model(**enc)
            last_pos = enc["attention_mask"].sum(dim=1) - 1
            last_logits = outputs.logits[
                torch.arange(enc["input_ids"].size(0)), last_pos
            ]
            label_logits = last_logits[:, [LABEL_IDS[0], LABEL_IDS[1]]]
            preds.extend(label_logits.argmax(-1).cpu().numpy())
    return preds

p_clean = predict_decoder(model, tok, X_te)
p_adv   = predict_decoder(model, tok, X_adv)
del model; torch.cuda.empty_cache()

r_clean = evaluate(y_te, p_clean, "Qwen2.5-1.5B+LoRA (clean test set)")
r_adv   = evaluate(y_te, p_adv,   "Qwen2.5-1.5B+LoRA (adversarial test set)")
drop    = float(r_clean["F1"]) - float(r_adv["F1"])

df_res = pd.DataFrame([r_clean, r_adv])
df_res["F1 drop"] = ["—", f"-{drop:.4f}"]

print("" + "="*60)
print("SCENARIO 6 — DECODER (ADVERSARIAL) RESULTS")
print("="*60)
print(df_res[["Model", "Accuracy", "F1", "F1 drop"]].to_string(index=False))
print(f"→ Qwen F1 drop: {drop:.4f}")

SCENARIO 6 — DECODER (ADVERSARIAL) RESULTS
                                   Model Accuracy     F1  F1 drop
      Qwen2.5-1.5B+LoRA (clean test set)   0.8309 0.7191        —
Qwen2.5-1.5B+LoRA (adversarial test set)   0.8342 0.7461 --0.0270
→ Qwen F1 drop: -0.0270


In [13]:
# ── Final side-by-side comparison ─────────────────────────────────────────────
qwen_drop = drop

# Try to load encoder results if the encoder notebook has been run
try:
    enc_res  = np.load("/kaggle/working/encoder_best_result.npy", allow_pickle=True).item()
    all_enc  = np.load("/kaggle/working/encoder_all_results.npy", allow_pickle=True).item()
    has_encoder_results = True
except FileNotFoundError:
    has_encoder_results = False

# ── Qwen standalone results ───────────────────────────────────────────────────
df_qwen = pd.DataFrame([
    {**r_clean, "F1 drop": "—"},
    {**r_adv,   "F1 drop": f"-{qwen_drop:.4f}"}
])
print("" + "="*60)
print("SCENARIO 6 — DECODER (ADVERSARIAL) RESULTS")
print("="*60)
print(df_qwen[["Model", "Accuracy", "F1", "F1 drop"]].to_string(index=False))
print(f"→ Qwen F1 drop: {qwen_drop:.4f}")

# ── Cross-notebook hypothesis test (only if encoder results exist) ────────────
if has_encoder_results:
    rows = []
    for name, res in all_enc.items():
        rows.append({
            "Model":    f"{name} (fine-tuned)",
            "Clean F1": f"{res['clean_f1']:.4f}",
            "Adv F1":   f"{res['adv_f1']:.4f}",
            "F1 Drop":  f"-{res['drop']:.4f}"
        })
    rows.append({
        "Model":    "Qwen2.5-1.5B (LoRA fine-tuned)",
        "Clean F1": r_clean["F1"],
        "Adv F1":   r_adv["F1"],
        "F1 Drop":  f"-{qwen_drop:.4f}"
    })
    comparison = pd.DataFrame(rows)

    print("" + "="*70)
    print("SCENARIO 6 — FINAL HYPOTHESIS TEST")
    print("BERT family (fine-tuned) vs Qwen2.5-1.5B (LoRA fine-tuned)")
    print("="*70)
    print(comparison.to_string(index=False))

    best_enc_drop = enc_res["drop"]
    best_enc_name = enc_res["name"]
    print(f"→ Best encoder: {best_enc_name}  (F1 drop: -{best_enc_drop:.4f})")
    print(f"→ Qwen2.5-1.5B: F1 drop: -{qwen_drop:.4f}")
    print()
    if qwen_drop < best_enc_drop:
        diff = best_enc_drop - qwen_drop
        print(f"✓ Hypothesis HOLDS — Qwen drops {diff:.4f} less than best encoder ({best_enc_name}).")
        print( "  Semantic reasoning (decoder) is more robust than token pattern matching (encoder).")
    elif abs(qwen_drop - best_enc_drop) < 0.005:
        print(f"~ Hypothesis INCONCLUSIVE — drops within 0.005 of each other.")
        print( "  Recommendation: repeat with full fine-tuning on A100 (40GB VRAM).")
    else:
        print(f"✗ Hypothesis FAILS — Qwen drops more than best encoder ({best_enc_name}).")
        print( "  Possible causes: LoRA under-adaptation, or hypothesis does not hold for this task.")
else:
    print("[INFO] Encoder notebook results not found.")
    print("  Run the encoder notebook to generate the full hypothesis test comparison.")
    print("  Qwen standalone results above are still valid.")


SCENARIO 6 — DECODER (ADVERSARIAL) RESULTS
                                   Model Accuracy     F1  F1 drop
      Qwen2.5-1.5B+LoRA (clean test set)   0.8309 0.7191        —
Qwen2.5-1.5B+LoRA (adversarial test set)   0.8342 0.7461 --0.0270
→ Qwen F1 drop: -0.0270
[INFO] Encoder notebook results not found.
  Run the encoder notebook to generate the full hypothesis test comparison.
  Qwen standalone results above are still valid.
